# 2. Predict: both models on every scene of the chosen blocks (GPU)

Only the model runs happen here. For each block the small camera layer is downloaded, VGGT-Omega and VGGT-1B
predict depth and camera pose for every scene, and the predictions are saved to persistent storage. Nothing is
scored, so this notebook reveals nothing about the results.

A GPU is needed only for this step, so it is kept apart from the scoring, which takes longer and runs on a CPU.
Measured on an A100: about 2 minutes per block of 20 scenes (the models run for 70 s of it; the next scene's
images are read in the background and both models stay loaded from block to block). Standard RAM is enough.

Safe to run again: a block whose predictions exist is skipped without a download.

**About the saved output below.** It is the record of the run that made the predictions, and it is older than two of the speed-ups this notebook now has: its full blocks took 229 to 246 s each, with the GPU waiting about 68 s per block for images and both models reloaded for every block. The same code path, run later for the test split (`05_test_predict_gpu`), took 124 to 139 s per block with a 3 s wait. The predictions themselves do not depend on any of this. Running this notebook again prints `already predicted` for all 15 blocks, because their predictions exist.

**Next:** remove the GPU server and run `03_score` with the same `BLOCKS`.

In [1]:
# --- 1. Configuration: use the SAME BLOCKS list afterwards in 03_score ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("train", 11), ("train", 12), ("train", 13),              # dark and wet: every scene is both (two recordings of one evening)
          ("train", 92), ("train", 82), ("train", 83), ("train", 90),   # motorway-rich
          ("train", 5), ("train", 6), ("train", 9),                 # wet in DAYLIGHT, to tell rain from darkness
          ("val", 0), ("val", 1), ("val", 2), ("val", 10), ("val", 12)]   # the validation blocks
# Chosen from the census for rare conditions, not at random. Finished blocks are skipped, so the list can grow.
# ("val", 11) is the development block (notebooks/development). The test blocks have their own notebooks, 05 to 07.

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)


In [4]:
# --- 4. Predict every block in the list ---
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
summaries = []
RUNNERS = {model: pl.ModelRunner(model) for model in MODELS}   # loaded once, kept for every block
for split, block in BLOCKS:
    assert (split, block) in available.index, f"block {(split, block)} is not downloadable with camera + LiDAR"
    print(f"=== {pl.block_tag(split, block)} ===")
    summary = pl.predict_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED), CAMERA, MODELS,
                               scene_names=ad.block_scene_names(scene_blocks, split, block, EXCLUDED), runners=RUNNERS)
    print(" ", summary)
    summaries.append(summary)
for runner in RUNNERS.values():
    runner.release()
print()
print(pd.DataFrame(summaries).to_string(index=False))
print()
print("Done. Remove the GPU server, then run 03_score on a CPU server with the same BLOCKS.")

scenes excluded by the dataset: 8
=== train_block000011 ===
  {'block': 'train_block000011', 'status': 'already predicted'}
=== train_block000012 ===
  {'block': 'train_block000012', 'status': 'already predicted'}
=== train_block000013 ===
  {'block': 'train_block000013', 'status': 'already predicted'}
=== train_block000092 ===
  {'block': 'train_block000092', 'status': 'already predicted'}
=== train_block000082 ===
  {'block': 'train_block000082', 'status': 'already predicted'}
=== train_block000083 ===
  {'block': 'train_block000083', 'status': 'already predicted'}
=== train_block000090 ===
  {'block': 'train_block000090', 'status': 'already predicted'}
=== train_block000005 ===
  downloading with the toolkit, decompressing with xz on all 12 cores


  fast unpack: {'archives': 2, 'xz_decompressed_on_all_cores': 0, 'download_s': 56.7, 'verify_and_decompress_s': 20.8, 'extract_s': 18.1}


  vggt_omega_512 loaded in 46 s
installing vggt


  vggt_1b loaded in 49 s


/usr/local/lib/python3.13/dist-packages/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


...

/usr/local/lib/python3.13/dist-packages/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  {'block': 'val_block000010', 'status': 'predicted', 'download_s': 62.6, 'model_seconds': 70.5, 'image_seconds': 68.1, 'save_seconds': 9.0, 'total_s': 237.3, 'vggt_omega_512_new': 20, 'vggt_1b_new': 20}
=== val_block000012 ===
  downloading with the toolkit, decompressing with xz on all 12 cores


  fast unpack: {'archives': 2, 'xz_decompressed_on_all_cores': 0, 'download_s': 5.9, 'verify_and_decompress_s': 4.6, 'extract_s': 3.8}
  vggt_omega_512 loaded in 14 s
  vggt_1b loaded in 11 s


/usr/local/lib/python3.13/dist-packages/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  {'block': 'val_block000012', 'status': 'predicted', 'download_s': 15.5, 'model_seconds': 24.7, 'image_seconds': 23.9, 'save_seconds': 3.1, 'total_s': 92.9, 'vggt_omega_512_new': 7, 'vggt_1b_new': 7}

            block            status  download_s  model_seconds  image_seconds  save_seconds  total_s  vggt_omega_512_new  vggt_1b_new
train_block000011 already predicted         NaN            NaN            NaN           NaN      NaN                 NaN          NaN
train_block000012 already predicted         NaN            NaN            NaN           NaN      NaN                 NaN          NaN
train_block000013 already predicted         NaN            NaN            NaN           NaN      NaN                 NaN          NaN
train_block000092 already predicted         NaN            NaN            NaN           NaN      NaN                 NaN          NaN
train_block000082 already predicted         NaN            NaN            NaN           NaN      NaN                 NaN        